# Shall we begin and create a baseline CNN model to see if those news are fake or real 😏

### First, some important imports

In [12]:
import numpy as np
import pickle
from pathlib import Path
from keras.models import Sequential # We'll use sequential layering, still need to konw why tbh.
from keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout # All the layers that we would probably need while builiding our model.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from keras.callbacks import EarlyStopping
from keras.layers import Input

### Some paths need to be defined first, so let's go ahead and define them 😉

In [6]:
BASE_DIR = Path.cwd().parents[1]
DATA_DIR = BASE_DIR / "data"
FEATURES_DIR = DATA_DIR / "dl_features"
print(BASE_DIR)
print(DATA_DIR)
print(FEATURES_DIR)

c:\Life\FCAI\Specialization - AI\Third_Year\2nd Semester\3 - Supervised Learning\7- Projects\nlp-fake-news-detector-transformers
c:\Life\FCAI\Specialization - AI\Third_Year\2nd Semester\3 - Supervised Learning\7- Projects\nlp-fake-news-detector-transformers\data
c:\Life\FCAI\Specialization - AI\Third_Year\2nd Semester\3 - Supervised Learning\7- Projects\nlp-fake-news-detector-transformers\data\dl_features


### Now we are finished with this paths thing, as always, some helper functions are needed, let's go and define them as well 😊

In [7]:
# A simple function to load the saved features that was done by my teammate.
def load_features():
    """Load the preprocessed arrays and metadata saved from the tokenization step. """
    print("Loading data...")
    X_train_pad = np.load(FEATURES_DIR / "X_train_pad.npy")
    X_val_pad = np.load(FEATURES_DIR / "X_val_pad.npy")
    X_test_pad = np.load(FEATURES_DIR / "X_test_pad.npy")

    y_train = np.load(FEATURES_DIR / "y_train.npy")
    y_val = np.load(FEATURES_DIR / "y_val.npy")
    y_test = np.load(FEATURES_DIR / "y_test.npy")

    with open(FEATURES_DIR / "meta.pkl", "rb") as f:
        meta = pickle.load(f)

    return X_train_pad, X_val_pad, X_test_pad, y_train, y_val, y_test, meta

### Time for the <b>Big Boss</b> has finally come. Let's make the *actual* model 😉

In [8]:
def build_cnn_model(max_words, max_len, embedding_dim = 50):
    """Build the 1D-CNN baseline model with randomly initialized embeddings."""
    print("Building CNN model....")
    model = Sequential([
        Embedding(input_dim = max_words, output_dim = embedding_dim, input_length = max_len),
        Conv1D(filters = 128, kernel_size = 5, activation = 'relu'),
        GlobalMaxPooling1D(),
        Dropout(0.5),
        Dense(1, activation = 'sigmoid')
    ])

    model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])
    return model

### Now that the model is built, it needs to be evaluated, ولا هو أي موديل وخلاص 🤨 

In [9]:
def evaluate_model(model, X_test, y_test):
    """"Evaluate the model using those metrics: Accuracy, Precision, Recall, and F1-score"""
    print("\nEvaluating the model on test data...")

    y_pred_probs = model.predict(X_test)
    y_pred_labels = (y_pred_probs > 0.5).astype(int)
    accuracy = accuracy_score(y_test, y_pred_labels)
    precision = precision_score(y_test, y_pred_labels)
    recall = recall_score(y_test, y_pred_labels)
    f1 = f1_score(y_test, y_pred_labels)

    print("\n--- Baseline 1D-CNN Results ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")

### Everything looks good until the moment you run it, the moment of truth, let's run all of what we have build so far and see how things turn up 👾

In [6]:
def main():

    X_train, X_val, X_test, y_train, y_val, y_test, meta = load_features()

    model = build_cnn_model(max_words = meta["max_words"], max_len = meta["max_len"])
    model.summary()

    print("\nStarting training...")
    history = model.fit(X_train, y_train, validation_data = (X_val, y_val), epochs = 5, batch_size = 32)

    evaluate_model(model, X_test, y_test)

    model.save(FEATURES_DIR / "cnn-baseline_model.h5")
    print("\nModel saved Successfully!")

if __name__ == "__main__":
    main()

Loading data...
Building CNN model....


C:\Users\Lenovo\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Starting training...
Epoch 1/5
30107/30107 ━━━━━━━━━━━━━━━━━━━━ 578s 19ms/step - accuracy: 0.7779 - loss: 0.4708 - val_accuracy: 0.7922 - val_loss: 0.4473
Epoch 2/5
30107/30107 ━━━━━━━━━━━━━━━━━━━━ 418s 14ms/step - accuracy: 0.7985 - loss: 0.4386 - val_accuracy: 0.7938 - val_loss: 0.4437
Epoch 3/5
30107/30107 ━━━━━━━━━━━━━━━━━━━━ 185s 6ms/step - accuracy: 0.8090 - loss: 0.4204 - val_accuracy: 0.7919 - val_loss: 0.4467
Epoch 4/5
30107/30107 ━━━━━━━━━━━━━━━━━━━━ 82s 3ms/step - accuracy: 0.8191 - loss: 0.4045 - val_accuracy: 0.7924 - val_loss: 0.4514
Epoch 5/5
30107/30107 ━━━━━━━━━━━━━━━━━━━━ 82s 3ms/step - accuracy: 0.8270 - loss: 0.3896 - val_accuracy: 0.7888 - val_loss: 0.4607

Evaluating the model on test data...
9264/9264 ━━━━━━━━━━━━━━━━━━━━ 7s 725us/step



--- Baseline 1D-CNN Results ---
Accuracy: 0.7874
Precision: 0.7685
Recall: 0.8154
F1-Score: 0.7913

Model saved Successfully!


### Ouch, this isn't that great, that some clear overfitting there, with some deprecated code that needs to be fixed, so let's tackle it one by one 😊

### Why not use some eraly stopping as a callback to save us from that malicious overfitting thingy ??

In [10]:
early_stopping = EarlyStopping(monitor = 'val_loss',
                               patience = 2,
                               restore_best_weights = True)

### Chaning our main function to utilize this trick 😉 (and change the model extension when we save it, cause the old one seems to be a legacy one)

In [11]:
def main():

    X_train, X_val, X_test, y_train, y_val, y_test, meta = load_features()

    model = build_cnn_model(max_words = meta["max_words"], max_len = meta["max_len"])
    model.summary()

    print("\nStarting training...")
    history = model.fit(X_train, y_train, validation_data = (X_val, y_val), epochs = 20, batch_size = 32, callbacks = [early_stopping])

    evaluate_model(model, X_test, y_test)

    model.save(FEATURES_DIR / "cnn-baseline_model.keras")
    print("\nModel saved Successfully!")

### Another error was in the build_cnn_model function, which cause the corruption of the summary table (used input_length arg. which is depricated, refering to an explicit input layer instead)

In [13]:
def build_cnn_model(max_words, max_len, embedding_dim = 50):
    """Build the 1D-CNN baseline model with randomly initialized embeddings."""
    print("Building CNN model....")
    model = Sequential([
        Input(shape = (max_len, )),
        Embedding(input_dim = max_words, output_dim = embedding_dim),
        Conv1D(filters = 128, kernel_size = 5, activation = 'relu'),
        GlobalMaxPooling1D(),
        Dropout(0.5),
        Dense(1, activation = 'sigmoid')
    ])

    model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])
    return model

In [14]:
if __name__ == "__main__":
    main()

Loading data...
Building CNN model....


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 50)         │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 16, 128)        │        32,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,032,257 (3.94 MB)

 Trainable params: 1,032,257 (3.94 MB)

 Non-trainable params: 0 (0.00 B)


Starting training...
Epoch 1/20
30107/30107 ━━━━━━━━━━━━━━━━━━━━ 88s 3ms/step - accuracy: 0.7780 - loss: 0.4711 - val_accuracy: 0.7922 - val_loss: 0.4479
Epoch 2/20
30107/30107 ━━━━━━━━━━━━━━━━━━━━ 84s 3ms/step - accuracy: 0.7992 - loss: 0.4387 - val_accuracy: 0.7940 - val_loss: 0.4455
Epoch 3/20
30107/30107 ━━━━━━━━━━━━━━━━━━━━ 83s 3ms/step - accuracy: 0.8091 - loss: 0.4210 - val_accuracy: 0.7936 - val_loss: 0.4449
Epoch 4/20
30107/30107 ━━━━━━━━━━━━━━━━━━━━ 83s 3ms/step - accuracy: 0.8189 - loss: 0.4044 - val_accuracy: 0.7907 - val_loss: 0.4567
Epoch 5/20
30107/30107 ━━━━━━━━━━━━━━━━━━━━ 101s 3ms/step - accuracy: 0.8272 - loss: 0.3900 - val_accuracy: 0.7894 - val_loss: 0.4602

Evaluating the model on test data...
9264/9264 ━━━━━━━━━━━━━━━━━━━━ 10s 1ms/step

--- Baseline 1D-CNN Results ---
Accuracy: 0.7935
Precision: 0.7806
Recall: 0.8098
F1-Score: 0.7949

Model saved Successfully!
